# Cell 1 — Install dependencies

In [1]:
!pip install tensorflow numpy

# Cell 2 — Import libraries

In [2]:
import numpy as np
import tensorflow as tf

# Cell 3 — Generate training data

In [3]:
SAMPLES = 5000

# Normal data range (20-30)
x_train = np.random.uniform(low=20.0, high=30.0, size=SAMPLES)
x_test  = np.random.uniform(low=20.0, high=30.0, size=int(SAMPLES * 0.2))

# Normalize
x_train_norm = (x_train - 20.0) / 10.0
x_test_norm  = (x_test - 20.0) / 10.0

print(f"Training samples: {len(x_train)}")
print(f"Test samples: {len(x_test)}")

Training samples: 5000
Test samples: 1000


# Cell 4 — Build and train model

In [4]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation='relu', input_shape=(1,)),
    tf.keras.layers.Dense(4, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.fit(x_train_norm, x_train_norm, epochs=100, validation_data=(x_test_norm, x_test_norm), verbose=0)
print("Training complete!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training complete!


# Cell 5 — Convert to TensorFlow Lite

In [5]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('anomaly_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"Model size: {len(tflite_model)} bytes")

Saved artifact at '/tmp/tmpswvwepmy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138328601210960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138328601212112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138328601210768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138328601209040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138328601212688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138328601209616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138328601213072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138328601211920: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model size: 2660 bytes


# Cell 6 — Convert to C array

In [6]:
!xxd -i anomaly_model.tflite > anomaly_model.h

with open('anomaly_model.h', 'r') as f:
    content = f.read()

content = content.replace('unsigned char anomaly_model_tflite[]', 'const unsigned char anomaly_model[]')
content = content.replace('unsigned int anomaly_model_tflite_len', 'const unsigned int anomaly_model_len')

with open('anomaly_model.h', 'w') as f:
    f.write(content)

print("anomaly_model.h generated!")

anomaly_model.h generated!


# Cell 7 — Download

In [7]:
from google.colab import files
files.download('anomaly_model.h')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>